# H1: 福岡県 深掘り分析

PBL発表 `PBL発表_v2.pptx` の13枚目「各地域の深掘り分析」に対応する、福岡県向けの追加分析ノートブック。

**目的**
- 福岡県に絞って、2018→2022の年別上昇率の推移を見る
- 県内の各路線価について、期間別の変化率を地図上で確認する
- 福岡中心部からの距離帯・地区区分ごとの違いを見る

**データ取り扱い**
- 実REXデータはライセンスデータのため、このノートブックではパスだけを指定する
- 生レコード表示（`head()` や行ダンプ）は行わない
- 出力は件数・平均・中央値・分布・図表に限定する


In [ ]:
import sys
from pathlib import Path

# notebook を repo root / notebooks のどちらから開いても src を読めるようにする
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "src").exists():
    SRC_DIR = PROJECT_ROOT / "src"
    OUT_DIR = PROJECT_ROOT / "outputs"
else:
    SRC_DIR = PROJECT_ROOT.parent / "src"
    OUT_DIR = PROJECT_ROOT.parent / "outputs"

sys.path.insert(0, str(SRC_DIR))
OUT_DIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm

try:
    import japanize_matplotlib
except ImportError:
    pass

from compare_years import build_panel
from compute_change import add_distance_to_city

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["axes.unicode_minus"] = False

RED = "#E8534A"
BLUE = "#3A7FC1"
GRAY = "#CCCCCC"
NAVY = "#1a1a2e"
GREEN = "#5BAD6F"
ORANGE = "#E8A838"


## 1. データパス設定

`PATH_2022` と `PATH_2020` を、手元のREX実データのパスに変更してから実行する。

- 2022年ファイル: 2020・2021・2022年価格を含む
- 2020年ファイル: 2018・2019・2020年価格を含む
- `build_panel()` が `serial_id` で結合し、2018〜2022年の5年パネルを作る


In [ ]:
# TODO: 手元の実データパスに変更してください
PATH_2022 = Path("/path/to/REX_data/2022/nouhin_line_2022.shp")
PATH_2020 = Path("/path/to/REX_data/2020/nouhin_line_2020.shp")

FUKUOKA_PREF_CODE = 40
FUKUOKA_CITY = "福岡"

# 期間ラベルと変化率カラム
PERIODS = [
    ("2018→2019", "chg_18_19", "コロナ前"),
    ("2019→2020", "chg_19_20", "発生年"),
    ("2020→2021", "chg_20_21", "コロナ禍"),
    ("2021→2022", "chg_21_22", "回復期"),
]


## 2. 5年パネル作成と福岡県抽出

ここでは件数のみ表示し、生レコードは表示しない。


In [ ]:
panel = build_panel(PATH_2022, PATH_2020)

fukuoka = panel[panel["pref_code"] == FUKUOKA_PREF_CODE].copy()
fukuoka = fukuoka.dropna(subset=[col for _, col, _ in PERIODS])

print(f"全国 5年パネル件数: {len(panel):,}")
print(f"福岡県 5年分有効路線数: {len(fukuoka):,}")
print(f"福岡県の全体変化率 中央値: {fukuoka['chg_total'].median():.2f}%")
print(f"福岡県のコロナ禍累計変化率 中央値: {fukuoka['chg_covid'].median():.2f}%")


## 3. 福岡県の年別上昇率推移

平均・中央値・下落割合を期間別に見る。


In [ ]:
rows = []
for label, col, phase in PERIODS:
    s = fukuoka[col].dropna()
    rows.append({
        "期間": label,
        "フェーズ": phase,
        "件数": int(s.count()),
        "平均(%)": round(s.mean(), 3),
        "中央値(%)": round(s.median(), 3),
        "上昇割合(%)": round((s > 0).mean() * 100, 1),
        "下落割合(%)": round((s < 0).mean() * 100, 1),
        "p25(%)": round(s.quantile(0.25), 3),
        "p75(%)": round(s.quantile(0.75), 3),
    })

fukuoka_yearly = pd.DataFrame(rows).set_index("期間")
fukuoka_yearly


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
labels = fukuoka_yearly.index.tolist()
means = fukuoka_yearly["平均(%)"].to_numpy()
medians = fukuoka_yearly["中央値(%)"].to_numpy()

bar_colors = [RED if v >= 0 else BLUE for v in means]
ax.bar(labels, means, color=bar_colors, alpha=0.72, width=0.54, label="平均")
ax.plot(labels, medians, color=NAVY, marker="o", linewidth=2.2, label="中央値")
ax.axhline(0, color="#666666", linewidth=1)

for i, v in enumerate(means):
    va = "bottom" if v >= 0 else "top"
    offset = 0.04 if v >= 0 else -0.04
    ax.text(i, v + offset, f"{v:.2f}%", ha="center", va=va, fontsize=10, fontweight="bold")

ax.set_title("福岡県: 年別路線価変化率の推移", fontsize=15, fontweight="bold")
ax.set_ylabel("変化率（%）")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()

out = OUT_DIR / "fukuoka_yearly_change.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 4. 地区区分別の推移

商業系・住宅系・工業系で、福岡県内の上昇率の出方を比較する。


In [ ]:
district_summary = (
    fukuoka.groupby("district_type")[[col for _, col, _ in PERIODS]]
    .agg(["count", "mean", "median"])
    .round(3)
)
district_summary


In [ ]:
plot_df = (
    fukuoka.groupby("district_type")[[col for _, col, _ in PERIODS]]
    .median()
    .rename(columns={col: label for label, col, _ in PERIODS})
)

fig, ax = plt.subplots(figsize=(9.5, 5.4))
for district in plot_df.index:
    ax.plot(plot_df.columns, plot_df.loc[district], marker="o", linewidth=2, label=district)

ax.axhline(0, color="#666666", linewidth=1)
ax.set_title("福岡県: 地区区分別の変化率中央値", fontsize=15, fontweight="bold")
ax.set_ylabel("中央値（%）")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=2)
plt.tight_layout()

out = OUT_DIR / "fukuoka_district_yearly_change.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 5. 路線単位の変化率分布

各期間について、福岡県内の路線ごとの変化率分布を見る。
外れ値で図が潰れないよう、表示範囲は1〜99パーセンタイルに制限する。


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
axes = axes.ravel()

for ax, (label, col, phase) in zip(axes, PERIODS):
    s = fukuoka[col].dropna()
    lo, hi = s.quantile([0.01, 0.99])
    ax.hist(s.clip(lo, hi), bins=60, color=NAVY, alpha=0.78)
    ax.axvline(s.median(), color=RED, linewidth=2, label=f"中央値 {s.median():.2f}%")
    ax.axvline(0, color="#666666", linewidth=1)
    ax.set_title(f"{label}（{phase}）", fontweight="bold")
    ax.set_xlabel("変化率（%）")
    ax.grid(axis="y", alpha=0.2)
    ax.legend(frameon=False, fontsize=9)

axes[0].set_ylabel("路線数")
axes[2].set_ylabel("路線数")
fig.suptitle("福岡県: 路線単位の変化率分布", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()

out = OUT_DIR / "fukuoka_change_distribution.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 6. 県内の各路線価変化率マップ

赤が上昇、青が下落。福岡県内の路線ごとの変化率を期間別に確認する。

※ 大量描画で重い場合は `MAX_PLOT` を小さくする。


In [ ]:
MAX_PLOT = 250_000

map_gdf = fukuoka.copy()
map_gdf["geometry"] = map_gdf.geometry.centroid
map_gdf = map_gdf.to_crs(epsg=3857)

if len(map_gdf) > MAX_PLOT:
    map_gdf = map_gdf.sample(MAX_PLOT, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, (label, col, phase) in zip(axes, PERIODS):
    sub = map_gdf.dropna(subset=[col])
    lim = max(abs(sub[col].quantile(0.02)), abs(sub[col].quantile(0.98)), 1.0)
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim)
    sub.plot(ax=ax, column=col, cmap="coolwarm", norm=norm, markersize=1.0, alpha=0.72, legend=False)
    ax.set_title(f"{label}（{phase}）", fontsize=12, fontweight="bold")
    ax.set_axis_off()
    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs=map_gdf.crs.to_string(), source=ctx.providers.CartoDB.Positron, zoom=9, alpha=0.55)
    except Exception:
        pass

# 共通カラーバー
sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.75, pad=0.02)
cbar.set_label("変化率（%）")
fig.suptitle("福岡県: 路線単位の変化率マップ", fontsize=16, fontweight="bold", y=0.94)

out = OUT_DIR / "fukuoka_road_change_maps.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 7. 福岡中心部からの距離帯別分析

13枚目の視点「人口増加 / アジア玄関口 / インバウンド需要の空間分布」に合わせ、福岡駅近辺を中心として距離帯別に変化率を比較する。


In [ ]:
fukuoka_dist = add_distance_to_city(fukuoka, FUKUOKA_CITY)

bins = [0, 2, 5, 10, 20, 40, 80, 160]
labels = [f"{bins[i]}-{bins[i+1]}km" for i in range(len(bins) - 1)]
fukuoka_dist["dist_band"] = pd.cut(
    fukuoka_dist[f"dist_{FUKUOKA_CITY}_km"],
    bins=bins,
    labels=labels,
    right=False,
)

dist_summary = (
    fukuoka_dist.groupby("dist_band", observed=True)[[col for _, col, _ in PERIODS] + ["chg_covid", "chg_total"]]
    .agg(["count", "median", "mean"])
    .round(3)
)
dist_summary


In [ ]:
dist_plot = (
    fukuoka_dist.groupby("dist_band", observed=True)[[col for _, col, _ in PERIODS]]
    .median()
    .rename(columns={col: label for label, col, _ in PERIODS})
)

fig, ax = plt.subplots(figsize=(10, 5.6))
for period in dist_plot.columns:
    ax.plot(dist_plot.index.astype(str), dist_plot[period], marker="o", linewidth=2, label=period)

ax.axhline(0, color="#666666", linewidth=1)
ax.set_title("福岡県: 福岡中心部からの距離帯別 変化率中央値", fontsize=15, fontweight="bold")
ax.set_xlabel("福岡中心部からの距離帯")
ax.set_ylabel("中央値（%）")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=2)
plt.xticks(rotation=25)
plt.tight_layout()

out = OUT_DIR / "fukuoka_distance_band_change.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 8. 路線タイプ分類

スライド用図と同じ考え方で、福岡県内の路線を5タイプに分類する。

- 構造的上昇: コロナ前・コロナ禍・回復期すべてプラス
- 回復型: コロナ禍に下落し、回復期にプラス
- コロナ型下落: コロナ禍に下落し、回復期も非プラス
- 構造的下落: コロナ前からマイナス
- 安定: 上記以外


In [ ]:
def classify_road(row):
    pre, covid, rec = row["chg_18_19"], row["chg_20_21"], row["chg_21_22"]
    if pd.isna(pre) or pd.isna(covid) or pd.isna(rec):
        return "不明"
    if pre < 0:
        return "構造的下落"
    if pre > 0 and covid > 0 and rec > 0:
        return "構造的上昇"
    if covid < 0:
        return "回復型" if rec > 0 else "コロナ型下落"
    return "安定"

fukuoka_type = fukuoka.copy()
fukuoka_type["road_type"] = fukuoka_type.apply(classify_road, axis=1)

type_summary = (
    fukuoka_type["road_type"]
    .value_counts()
    .rename_axis("路線タイプ")
    .to_frame("件数")
)
type_summary["構成比(%)"] = (type_summary["件数"] / type_summary["件数"].sum() * 100).round(1)
type_summary


In [ ]:
TYPE_COLORS = {
    "構造的上昇": RED,
    "回復型": GREEN,
    "コロナ型下落": ORANGE,
    "構造的下落": BLUE,
    "安定": "#777777",
    "不明": GRAY,
}

fig, ax = plt.subplots(figsize=(8.5, 5.2))
plot_order = [t for t in ["構造的上昇", "回復型", "コロナ型下落", "構造的下落", "安定", "不明"] if t in type_summary.index]
vals = type_summary.loc[plot_order, "構成比(%)"]
colors = [TYPE_COLORS[t] for t in plot_order]

ax.barh(plot_order, vals, color=colors, alpha=0.82)
for i, v in enumerate(vals):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=10, fontweight="bold")

ax.set_title("福岡県: 路線タイプ構成", fontsize=15, fontweight="bold")
ax.set_xlabel("構成比（%）")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()

out = OUT_DIR / "fukuoka_road_type_share.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


In [ ]:
type_map = fukuoka_type[fukuoka_type["road_type"].isin(TYPE_COLORS)].copy()
type_map["geometry"] = type_map.geometry.centroid
type_map = type_map.to_crs(epsg=3857)

if len(type_map) > MAX_PLOT:
    type_map = type_map.sample(MAX_PLOT, random_state=42)

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor("#f7f7f7")

# 背景に安定、前面に特徴的タイプ
draw_order = ["安定", "回復型", "コロナ型下落", "構造的下落", "構造的上昇"]
for t in draw_order:
    sub = type_map[type_map["road_type"] == t]
    if sub.empty:
        continue
    size = 0.5 if t == "安定" else 1.3
    alpha = 0.18 if t == "安定" else 0.75
    sub.plot(ax=ax, color=TYPE_COLORS[t], markersize=size, alpha=alpha)

try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=type_map.crs.to_string(), source=ctx.providers.CartoDB.Positron, zoom=9, alpha=0.55)
except Exception:
    pass

handles = [
    mlines.Line2D([], [], color=TYPE_COLORS[t], marker="o", ls="none", markersize=7,
                  label=f"{t} {type_summary.loc[t, '構成比(%)']:.1f}%")
    for t in draw_order if t in type_summary.index
]
ax.legend(handles=handles, loc="lower left", frameon=True, facecolor="white", framealpha=0.88)
ax.set_title("福岡県: 路線タイプ分布", fontsize=15, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()

out = OUT_DIR / "fukuoka_road_type_map.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"保存: {out}")


## 9. スライド13用の読み取りメモ

以下の観点で、福岡の分析コメントを作る。

1. 年別推移: コロナ前から上昇基調か、コロナ禍で鈍化したか、2021→2022で回復したか
2. 空間分布: 上昇路線が福岡中心部・湾岸・交通結節点周辺に集中しているか
3. 距離帯: 中心部から何km圏で上昇率が高いか
4. 地区区分: 商業系・住宅系のどちらが強いか
5. 路線タイプ: 「構造的上昇」「回復型」が県内のどこに分布するか

生成されたPNGは `outputs/` に保存されるので、PPTXの13枚目に差し込める。
